**Requerimientos**


In [0]:
%pip install pyyaml

**Cargando config.yml**

In [0]:
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import lit, col, to_timestamp
import yaml

with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)
print("✅ Configuración cargada")

**Creando esquema silver**

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['spark']['database']}.silver")
print(f"✅ Esquema {config['spark']['database']}.silver creado")

**Leyendo tabla bronze**

In [0]:
bronze_df = spark.table(config["tables"]["bronze_table"])
display(bronze_df.limit(5))
print("✅ Datatable cargada")

**Creando las siguientes transformaciones:**
- Limpia valores nulos. 
- Convierte tipos de datos (Strings a Timestamps/Integers). 
- Filtra datos irrelevantes.

In [0]:
# Limpia valores nulos
bronze_df_clean = bronze_df.dropna()

# Convierte tipos de datos
bronze_df_clean = bronze_df_clean \
    .withColumn("Date_Time", to_timestamp(col("Date_Time"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("Temperature_C", col("Temperature_C").cast("double")) \
    .withColumn("Humidity_pct", col("Humidity_pct").cast("double")) \
    .withColumn("Precipitation_mm", col("Precipitation_mm").cast("double")) \
    .withColumn("Wind_Speed_kmh", col("Wind_Speed_kmh").cast("double"))

# Filtra datos relevantes
bronze_df_clean = bronze_df_clean.filter(
    (col("Temperature_C") > -50) & (col("Temperature_C") < 60) &
    (col("Humidity_pct") >= 0) & (col("Humidity_pct") <= 100) &
    (col("Precipitation_mm") >= 0) &
    (col("Wind_Speed_kmh") >= 0)
)

display(bronze_df_clean.limit(5))
print(f"✅ {bronze_df_clean.count()} registros validos")

In [0]:
bronze_df_clean.write.format("delta") \
    .mode(config["spark"]["write_mode"]) \
    .saveAsTable(config["tables"]["silver_table"])

print(f"✅ Dataset guardado en {config['tables']['silver_table']}")
display(spark.sql(f"DESCRIBE DETAIL {config['tables']['silver_table']}"))